<a href="https://colab.research.google.com/github/parikhv04/LexisNexis-1A-detecting-fraudulent-insurance-claims/blob/main/LexisNexis1A_M2_Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LexisNexis Fraud Detection Project

Break Through Tech AI Studio - Fall 2026

This notebook contains advanced feature engineering using the cleaned dataset from M1.

Task 1 Feature Engineering and Preprocessing Summary: We carried forward the 26 features from September and created additional features describing age groups, approximate vehicle age, claim reporting lag, and weekend timing. We also grouped infrequent levels of higher-cardinality categorical variables during preprocessing. The preprocessing pipeline performs median imputation and scaling for numeric variables and one-hot encoding for categorical variables. All learned preprocessing parameters, including medians, scaling parameters, and rare-category groupings, are fit using the 1994 training set only and then applied unchanged to the 1995 validation and 1996 test sets.


In [ ]:
import pandas as pd
import numpy as np

!git clone https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git

%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

df = pd.read_csv("data/fraud_oracle_cleaned.csv")

TARGET = "FraudFound_P"

SAFE_FEATURES = [
    "Month",
    "WeekOfMonth",
    "DayOfWeek",
    "MonthClaimed",
    "WeekOfMonthClaimed",
    "DayOfWeekClaimed",
    "Make",
    "VehicleCategory",
    "VehiclePrice",
    "AgeOfVehicle",
    "NumberOfCars",
    "AccidentArea",
    "Fault",
    "Sex",
    "MaritalStatus",
    "Age",
    "Age_missing",
    "AgeOfPolicyHolder",
    "PolicyType",
    "BasePolicy",
    "Deductible",
    "DriverRating",
    "AgentType",
    "Days_Policy_Accident",
    "Days_Policy_Claim",
    "PastNumberOfClaims"
]

Cloning into 'LexisNexis-1A-detecting-fraudulent-insurance-claims'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 117 (delta 70), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 1.35 MiB | 3.87 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [ ]:
train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
test  = df[df["Year"] == 1996].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (6141, 34)
Validation: (5195, 34)
Test: (4083, 34)


##Here is feature engineering:

In [ ]:
MONTH_NUM = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

VEHICLE_AGE_MAP = {
    "new": 0,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "more than 7": 8
}


def add_engineered_features(data):
    data = data.copy()

    # Age groups derived from the cleaned numeric Age field
    data["AgeBand"] = pd.cut(
        data["Age"],
        bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
    )

    # Preserve missing Age explicitly rather than assigning it an age band
    data["AgeBand"] = data["AgeBand"].astype("object").fillna("Missing")

    # Convert vehicle-age bucket into an approximate ordered numeric feature
    data["VehicleAgeApprox"] = data["AgeOfVehicle"].map(VEHICLE_AGE_MAP)

    # Approximate number of months between accident and claim filing
    accident_month = data["Month"].map(MONTH_NUM)
    claimed_month = data["MonthClaimed"].map(MONTH_NUM)

    data["ClaimLagMonths"] = (
        claimed_month - accident_month
    ) % 12

    # Simple timing features
    data["WeekendAccident"] = (
        data["DayOfWeek"].isin(["Saturday", "Sunday"])
    ).astype(int)

    data["WeekendClaim"] = (
        data["DayOfWeekClaimed"].isin(["Saturday", "Sunday"])
    ).astype(int)

    return data

In [ ]:
X_train = add_engineered_features(train[SAFE_FEATURES])
X_val   = add_engineered_features(val[SAFE_FEATURES])
X_test  = add_engineered_features(test[SAFE_FEATURES])

y_train = train[TARGET]
y_val   = val[TARGET]
y_test  = test[TARGET]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
numeric_features = (
    X_train.select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train.select_dtypes(exclude="number")
    .columns
    .tolist()
)

In [ ]:
grouped_categories = [
    "Make",
    "PolicyType"
]

regular_categories = [
    col for col in categorical_features
    if col not in grouped_categories
]

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

In [ ]:
grouped_category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=0.01
    ))
])

In [ ]:
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        regular_categories
    ),
    (
        "grouped_categorical",
        grouped_category_pipeline,
        grouped_categories
    )
])

In [ ]:
preprocessor.fit(X_train)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['WeekOfMonth', 'WeekOfMonthClaimed', 'Age',
                                  'Age_missing', 'Deductible', 'DriverRating',
                                  'VehicleAgeApprox', 'ClaimLagMonths',
                                  'WeekendAccident', 'WeekendClaim']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strat...
                                  'NumberOfCars', 'AccidentArea', 'Fault',
                                  'Sex', 'MaritalStatus', 'AgeOfPolicyHolder',
                                  'BasePolicy', 'AgentType',
                                  'Days_Policy_Accident', 'Days_Policy_Claim',
                                  'PastNumberOfClaims', 'AgeBand']),
                                ('grouped_categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=0.01))]),
                                 ['Make', 'PolicyType'])])

In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

In [ ]:
print("Raw train shape:", X_train.shape)
print("Processed train shape:", X_train_processed.shape)

print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Raw train shape: (6141, 31)
Processed train shape: (6141, 128)
Processed validation shape: (5195, 128)
Processed test shape: (4083, 128)


In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

feature_names[:30]

Number of processed features: 128


array(['numeric__WeekOfMonth', 'numeric__WeekOfMonthClaimed',
       'numeric__Age', 'numeric__Age_missing', 'numeric__Deductible',
       'numeric__DriverRating', 'numeric__VehicleAgeApprox',
       'numeric__ClaimLagMonths', 'numeric__WeekendAccident',
       'numeric__WeekendClaim', 'categorical__Month_Apr',
       'categorical__Month_Aug', 'categorical__Month_Dec',
       'categorical__Month_Feb', 'categorical__Month_Jan',
       'categorical__Month_Jul', 'categorical__Month_Jun',
       'categorical__Month_Mar', 'categorical__Month_May',
       'categorical__Month_Nov', 'categorical__Month_Oct',
       'categorical__Month_Sep', 'categorical__DayOfWeek_Friday',
       'categorical__DayOfWeek_Monday', 'categorical__DayOfWeek_Saturday',
       'categorical__DayOfWeek_Sunday', 'categorical__DayOfWeek_Thursday',
       'categorical__DayOfWeek_Tuesday',
       'categorical__DayOfWeek_Wednesday',
       'categorical__MonthClaimed_Apr'], dtype=object)